# Heart Disease Prediction - End-to-End Notebook
Run top to bottom in Google Colab. Steps: load data, EDA, preprocessing, model comparison, evaluation, SHAP explainability, save model.

**Colab tip:** the dataset is downloaded from GitHub in the first cell, so no upload is needed.

In [ ]:
!pip install -q xgboost shap
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, joblib, warnings
warnings.filterwarnings('ignore')
URL = 'https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv'
df = pd.read_csv(URL)
df.columns = [c.strip().lstrip('\ufeff') for c in df.columns]
print(df.shape); df.head()

## 1. Exploratory Data Analysis

In [ ]:
print(df.info())
print('Duplicates:', df.duplicated().sum())
print('Missing:', df.isna().sum().sum())
df.describe().T

In [ ]:
df = df.drop_duplicates()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df['target'].value_counts().plot.bar(ax=ax[0], color=['#2980b9', '#c0392b'])
ax[0].set_title('Target distribution (1 = disease)')
sns.heatmap(df.corr(), cmap='coolwarm', center=0, ax=ax[1]); ax[1].set_title('Correlation heatmap')
plt.tight_layout(); plt.show()

In [ ]:
num = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
df[num + ['target']].groupby('target').mean().T.plot.bar(figsize=(7, 4), title='Mean of numeric features by target')
plt.show()
sns.countplot(data=df, x='cp', hue='target'); plt.title('Chest pain type vs disease'); plt.show()

## 2. Preprocessing and split

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
NUMERIC = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CATEGORICAL = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
X, y = df.drop(columns='target'), df['target']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
def build(model):
    pre = ColumnTransformer([('num', StandardScaler(), NUMERIC), ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)])
    return Pipeline([('pre', pre), ('model', model)])

## 3. Train and tune models
We tune for **recall** because in medical screening, missing a sick patient is the costly error.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
cv = StratifiedKFold(5, shuffle=True, random_state=42)
cands = {
 'Logistic Regression': (LogisticRegression(max_iter=2000), {'model__C': [0.01, 0.1, 1, 10]}),
 'Random Forest': (RandomForestClassifier(random_state=42), {'model__n_estimators': [200, 400], 'model__max_depth': [3, 5, None], 'model__min_samples_leaf': [1, 3]}),
 'SVM': (SVC(probability=True, random_state=42), {'model__C': [0.1, 1, 10], 'model__kernel': ['rbf', 'linear']}),
 'XGBoost': (XGBClassifier(eval_metric='logloss', random_state=42), {'model__n_estimators': [100, 300], 'model__max_depth': [2, 3, 4], 'model__learning_rate': [0.03, 0.1]}),
}
rows, fitted = [], {}
for name, (m, grid) in cands.items():
    gs = GridSearchCV(build(m), grid, cv=cv, scoring='recall', n_jobs=-1).fit(X_tr, y_tr)
    est = gs.best_estimator_; pred = est.predict(X_te); proba = est.predict_proba(X_te)[:, 1]
    rows.append(dict(model=name, accuracy=accuracy_score(y_te, pred), precision=precision_score(y_te, pred), recall=recall_score(y_te, pred), f1=f1_score(y_te, pred), roc_auc=roc_auc_score(y_te, proba)))
    fitted[name] = est
results = pd.DataFrame(rows).sort_values(['recall', 'roc_auc'], ascending=False).round(3)
results

## 4. Evaluation: ROC curves and confusion matrix

In [ ]:
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay, classification_report
best_name = results.iloc[0]['model']; best = fitted[best_name]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for n, e in fitted.items(): RocCurveDisplay.from_estimator(e, X_te, y_te, ax=ax[0], name=n)
ax[0].plot([0, 1], [0, 1], 'k--', alpha=.4); ax[0].set_title('ROC curves')
ConfusionMatrixDisplay.from_estimator(best, X_te, y_te, ax=ax[1], cmap='Blues'); ax[1].set_title(f'Confusion matrix - {best_name}')
plt.tight_layout(); plt.show()
print(classification_report(y_te, best.predict(X_te), target_names=['No disease', 'Disease']))

## 5. Explainability with SHAP
Shows which features push an individual prediction toward or away from heart disease.

In [ ]:
import shap
model_step = best.named_steps['model']
pre = best.named_steps['pre']
Xt = pre.transform(X_te); Xt = Xt.toarray() if hasattr(Xt, 'toarray') else Xt
names = pre.get_feature_names_out()
if best_name in ('Random Forest', 'XGBoost'):
    sv = shap.TreeExplainer(model_step).shap_values(Xt)
    sv = sv[1] if isinstance(sv, list) else (sv[..., 1] if sv.ndim == 3 else sv)
else:
    bg = shap.sample(Xt, 50, random_state=42)
    sv = shap.KernelExplainer(lambda d: model_step.predict_proba(d)[:, 1], bg).shap_values(Xt[:40])
    Xt = Xt[:40]
shap.summary_plot(sv, Xt, feature_names=names)

## 6. Save the model
Download `best_model.joblib` from the Colab file panel and place it in the `models/` folder of the project to use the Streamlit app.

In [ ]:
NUM_CAT = NUMERIC + CATEGORICAL
joblib.dump({'pipeline': best, 'name': best_name, 'features': NUM_CAT}, 'best_model.joblib')
try:
    from google.colab import files; files.download('best_model.joblib')
except ImportError:
    pass

## 7. Conclusions
- Best model by recall: *fill in from the table above*
- Top features: *fill in from SHAP*
- Limitations: small dataset (about 300 rows), single source, no external validation. Not a clinical tool.
- Future work: combine multiple datasets, calibration, deploy on Streamlit Cloud.